# Real-Time Facial Expression Recognition

This notebook loads an older Teachable Machine `keras_model.h5` using legacy `tf_keras` and runs webcam prediction.

Files expected in `C:\Deep Learning\PR-4`:
- `keras_model.h5`
- `labels.txt`


In [ ]:
%pip install -U "tf-keras==2.21.0" opencv-python numpy


**After the installation cell finishes, restart the Jupyter kernel once, then continue.**

In [2]:
import os
import numpy as np
import cv2
import tensorflow as tf
import tf_keras
from tf_keras.models import load_model

print('TensorFlow:', tf.__version__)
print('tf_keras:', tf_keras.__version__)
print('OpenCV:', cv2.__version__)



TensorFlow: 2.21.0
tf_keras: 2.21.0
OpenCV: 5.0.0


In [3]:
BASE_DIR = r'C:\Deep Learning\PR-4'
MODEL_PATH = os.path.join(BASE_DIR, 'keras_model.h5')
LABELS_PATH = os.path.join(BASE_DIR, 'labels.txt')

if not os.path.isfile(MODEL_PATH):
    raise FileNotFoundError(MODEL_PATH)
if not os.path.isfile(LABELS_PATH):
    raise FileNotFoundError(LABELS_PATH)

with open(LABELS_PATH, 'r', encoding='utf-8') as f:
    class_names = [line.strip() for line in f if line.strip()]

print('Classes:', class_names)


Classes: ['0 Angry', '1 Disgust', '2 Fear', '3 Happy', '4 Neutral', '5 sad', '6 Surprised']


In [4]:
print('Loading model...')
model = load_model(MODEL_PATH, compile=False)
print('✅ Model loaded successfully!')
model.summary()


Loading model...


✅ Model loaded successfully!
Model: "sequential_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 sequential_1 (Sequential)   (None, 1280)              410208    
                                                                 
 sequential_3 (Sequential)   (None, 7)                 128800    
                                                                 
Total params: 539008 (2.06 MB)
Trainable params: 524928 (2.00 MB)
Non-trainable params: 14080 (55.00 KB)
_________________________________________________________________


## Webcam prediction

Press **Q** or **ESC** to stop.

In [5]:
CAMERA_INDEX = 0
WIDTH = 224
HEIGHT = 224

camera = cv2.VideoCapture(CAMERA_INDEX)
if not camera.isOpened():
    raise RuntimeError('Could not open webcam. Try CAMERA_INDEX = 1.')

print('✅ Camera started. Press Q or ESC to quit.')

while True:
    ret, frame = camera.read()
    if not ret:
        print('Could not read webcam frame.')
        break

    frame = cv2.flip(frame, 1)
    display = frame.copy()

    image = cv2.resize(frame, (WIDTH, HEIGHT), interpolation=cv2.INTER_AREA)
    image = np.asarray(image, dtype=np.float32).reshape(1, HEIGHT, WIDTH, 3)
    image = (image / 127.5) - 1.0

    prediction = model.predict(image, verbose=0)
    index = int(np.argmax(prediction[0]))
    confidence = float(prediction[0][index]) * 100

    label = class_names[index] if index < len(class_names) else f'Class {index}'
    if len(label) > 2 and label[0].isdigit() and label[1].isspace():
        label = label[2:].strip()

    cv2.rectangle(display, (10, 10), (560, 105), (0, 0, 0), -1)
    cv2.putText(display, f'Expression: {label}', (25, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
    cv2.putText(display, f'Confidence: {confidence:.1f}%', (25, 85), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    cv2.imshow('Facial Expression Recognition', display)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q') or key == 27:
        break

camera.release()
cv2.destroyAllWindows()
print('✅ Camera stopped.')


✅ Camera started. Press Q or ESC to quit.
✅ Camera stopped.
